In [ ]:
# @author: lajello
# A collection of examples of how to use APIs and API wrappers.

### Imports

In [ ]:
# generic imports useful for all APIs
import requests
import json
import os
import time
import datetime
import pandas as pd
import bz2
import shutil
from pprint import pprint

requests.__version__
import platform
print(platform.python_version())

### TMDB API

Create an account and get an access token here: https://www.themoviedb.org/subscribe/developer

Check the documentation for the signatures of different calls: https://developer.themoviedb.org/reference/intro/getting-started


In [ ]:
# ADD VALUES HERE
# the API access tokens
tmdb_access_token = ''
tmdb_api_key = ''

In [ ]:
class TMDB:
    """
    Simple class to handle requests to the TMDB API
    https://developer.themoviedb.org/reference
    """
    def __init__(self, access_token, api_key=None, delay=0.5):
        """
        @param access_token copied from the TMBD API registration procedure
        @param api_key copied from the TMBD API registration procedure (not used, access_token is sufficient)
        @param delay in seconds between requests. Mind rate limits: https://developer.themoviedb.org/docs/rate-limiting
        """
        self.endpoint = 'https://api.themoviedb.org/3/'
        self.pictures_endpoint = 'https://image.tmdb.org/t/p/w470_and_h470_face'
        self.access_token = access_token
        self.api_key = api_key
        self.delay = delay
        self.last_request = 0 # timestamp of last request sent

    def update_delay(delay):
        """
        Changes the waiting delay (in seconds)·
        Mind rate limits: https://developer.themoviedb.org/docs/rate-limiting
        """
        self.delay = delay

    def get(self, resource, params={}):
        """
        Submits a request to the API
        @param resource: the part of the url to be attached to the endpoint to request the desired resource
        @param params: a dictionary of parameters
        """
        # rudimentary waiting mechanism not to go beyond the rate limits
        ts = datetime.datetime.now().timestamp()
        ts_delta = ts - self.last_request
        if ts_delta < self.delay:
            time.sleep(self.delay - ts_delta)
        self.last_request = ts

        response = requests.get(self.endpoint+'/'+resource,
                                headers={"Authorization":f"Bearer {tmdb_access_token}"},
                                params=params)
        http_status = response.status_code # should be 200
        http_reason = response.reason # should be 'OK'
        json_response = json.loads(response.content.decode('utf8'))

        return json_response

    def get_image(self, image_name, save_path=None):
        """
        Download image directly without the use of the API
        @param image_name: image file name complete with extension
        """
        pic_url = f'{self.pictures_endpoint}/{image_name}'
        if not save_path:
            save_path = f'./{image_name}'
        image = requests.get(pic_url, stream=True)
        # save image on file
        with open(save_path, 'wb') as out_file:
            shutil.copyfileobj(image.raw, out_file)

In [ ]:
# initialize API object
tmdb_api = TMDB(tmdb_access_token)

#### Network of actors and movies

In [ ]:
# search a movie to its TMDB id
movie_search_result = tmdb_api.get('search/movie',{'query':'eternal sunshine of the spotless mind'})
movie_id = movie_search_result['results'][0]['id']
movie_title = movie_search_result['results'][0]['title']
print(movie_id, movie_title)

In [ ]:
# use the movie id to get detailed information about the movie
movie_search_result = tmdb_api.get(f'/movie/{movie_id}')
movie_overview = movie_search_result['overview']

In [ ]:
print(movie_overview)

In [ ]:
# get movie cast
movie_credits_result = tmdb_api.get(f'/movie/{movie_id}/credits')
movie_cast = movie_credits_result['cast']
actor_ids = []
for person in movie_cast:
    person_id = person['id']
    person_occupation = person['known_for_department']
    if person_occupation == 'Acting':
        actor_ids.append(person_id)
print(f'Found {len(actor_ids)} actors in the movie {movie_title}')

In [ ]:
for actor_id in actor_ids:
    # get information on the actor
    person_results = tmdb_api.get(f'/person/{actor_id}')
    person_biography = person_results['biography']
    person_name = person_results['name']
    tmdb_api.get_image(person_results['profile_path'], f'tmdb_pics/{person_name}.jpg')

    # get names of movies the actor starred in
    person_credits_results = tmdb_api.get(f'/person/{actor_id}/movie_credits')
    movie_ids = []
    movie_titles = []
    for movie in person_credits_results['cast']:
        if movie['popularity'] > 7 and movie['vote_average'] > 7: # arbitrary popularity thresholds
            movie_ids.append(movie['id'])
            movie_titles.append(movie['title'])
    print(f'Actor {person_name} starred in popular movies: {", ".join(movie_titles)}')

#### Pagination

In [ ]:
# search all movies containing the word "the"
movie_search_result = tmdb_api.get('search/movie',{'query':'the'})
page_number = movie_search_result['page']
print(f'This is page {page_number} of many')
original_title = movie_search_result['results'][0]['original_title']
print(f'First results in page {page_number}: {original_title}')

In [ ]:
# move to the next page
movie_search_result = tmdb_api.get('search/movie',{'query':'the', 'page':2})
page_number = movie_search_result['page']
print(f'This is page {page_number} of many')
original_title = movie_search_result['results'][0]['original_title']
print(f'First results in page {page_number}: {original_title}')

### Reddit API

#### OAuth authentication
Got to https://www.reddit.com/prefs/apps and create a new application 

In [ ]:
# ADD VALUES HERE
# get these values by registering an application at https://www.reddit.com/prefs/apps
personal_use_script = ''
secret = ''

# create an object to be used for http basic authentication
auth = requests.auth.HTTPBasicAuth(personal_use_script, secret)

# ADD VALUES HERE
# username and password to be placed in the token request
data = {'grant_type': 'password',
        'username': '',
        'password': ''}

# setup our header info, which gives reddit a brief description of our app
headers = {'User-Agent': '/u/yourusername API python tutorial bot'}

# send a POST request for an OAuth token
response = requests.post('https://www.reddit.com/api/v1/access_token',
                         auth=auth, data=data, headers=headers)

print(response.json())
# get the access token from the response
oauth_token = response.json()['access_token']

# add the token to the http request header
# the ** operator map the keys of the two dictionaries and add the values
# of the second dict to the first dict
headers = {**headers, **{'Authorization': f'bearer {oauth_token}'}}

# send a request for a resource using the authentication header
response = requests.get('https://oauth.reddit.com/r/wallstreetbets/new', headers=headers)
json_response = response.json()
json_response

#### PRAW (Python Reddit API Wrapper)

In [ ]:
# https://praw.readthedocs.io/en/stable/getting_started/quick_start.html
import praw

reddit = praw.Reddit(
    client_id="",
    client_secret="",
    password="",
    user_agent="",
    username="",
)

subreddit = reddit.subreddit("wallstreetbets")
print(subreddit.title)
print(subreddit.description)